In [1]:
import keras
from keras import layers

inputs = layers.Input(shape=(224, 224, 3))
x = layers.ZeroPadding2D(padding=3)(inputs)
x = layers.Conv2D(64, 7, strides=2) (x)
x = layers.BatchNormalization(epsilon=1e-5) (x)
x = layers.Activation('relu') (x)
x = layers.ZeroPadding2D(padding=1) (x)
x = layers.MaxPooling2D(3, strides=2) (x)

2026-04-14 15:06:55.787690: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-14 15:06:55.834236: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-14 15:06:55.834282: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-14 15:06:55.837963: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-14 15:06:55.855181: I tensorflow/core/platform/cpu_feature_guar

In [2]:
def build_stack(x):
    x = residual_stack(x, 3, 64, first_stride = 1)
    for blocks, filters in [ (4, 128), (6, 256), (3, 512)]:
        x = residual_stack(x, blocks, filters, first_stride = 2)
    return x

def residual_stack(x, blocks, filters, first_stride=2):
    x = residual_block(x, filters, first_stride = first_stride, conv_skip=True)
    for _ in range(1, blocks):
        x = residual_block(x, filters, first_stride =1, conv_skip=False)
    return x

In [3]:
def residual_block(x, filters, first_stride=1, conv_skip=False):
    skip_conn = x
    x = layers.Conv2D(filters=filters, kernel_size=1, strides=first_stride) (x)
    x = layers.BatchNormalization(epsilon=1e-5) (x)
    x = layers.Activation('relu') (x)

    x = layers.Conv2D(filters=filters, kernel_size=3, padding='same') (x)
    x = layers.BatchNormalization(epsilon=1e-5) (x)
    x = layers.Activation('relu') (x)

    x = layers.Conv2D(filters=filters*4, kernel_size=1) (x)
    x = layers.BatchNormalization(epsilon=1e-5) (x)

    if conv_skip == True:
        skip_conn = layers.Conv2D(filters=filters * 4, kernel_size=1, 
                                  strides=first_stride) (skip_conn)
        skip_conn = layers.BatchNormalization(epsilon=1e-5) (skip_conn)
    x = layers.Add()([skip_conn , x])
    x = layers.Activation('relu') (x)
    return x

In [4]:
x = build_stack(x)

In [5]:
x = layers.GlobalAveragePooling2D() (x)
outputs = layers.Dense( 1000 , activation='softmax') (x)

In [6]:
model = keras.Model( inputs , outputs )

In [7]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 zero_padding2d (ZeroPaddin  (None, 230, 230, 3)          0         ['input_1[0][0]']             
 g2D)                                                                                             
                                                                                                  
 conv2d (Conv2D)             (None, 112, 112, 64)         9472      ['zero_padding2d[0][0]']      
                                                                                                  
 batch_normalization (Batch  (None, 112, 112, 64)         256       ['conv2d[0][0]']          

In [8]:
from PIL import Image
import numpy as np
from keras.applications import resnet

dog_png = Image.open('images/dog.png')
# display(dog_png)
res_dog_png = resnet.preprocess_input( np.array(dog_png) )

In [9]:
resnet50 = keras.applications.ResNet50()
predictions = resnet50.predict(res_dog_png[np.newaxis, : ])
resnet.decode_predictions( predictions )

2026-04-14 15:06:59.232000: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2026-04-14 15:06:59.310455: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


1/1 [==============================] - 1s 1s/step


[[('n02099712', 'Labrador_retriever', 0.3852792),
  ('n02099601', 'golden_retriever', 0.08966382),
  ('n02100735', 'English_setter', 0.04208234),
  ('n02106166', 'Border_collie', 0.037757363),
  ('n02101388', 'Brittany_spaniel', 0.030680671)]]

In [10]:
cat_png = Image.open('images/cat.png')
# display(dog_png)
res_cat_png = resnet.preprocess_input( np.array(cat_png) )
resnet50 = keras.applications.ResNet50()
predictions = resnet50.predict(res_cat_png[np.newaxis, : ])
resnet.decode_predictions( predictions )

1/1 [==============================] - 0s 387ms/step


[[('n02123045', 'tabby', 0.86867154),
  ('n02124075', 'Egyptian_cat', 0.050775655),
  ('n02123159', 'tiger_cat', 0.042543925),
  ('n07930864', 'cup', 0.0027601307),
  ('n03443371', 'goblet', 0.002097386)]]

In [11]:
from keras.applications import inception_v3
from keras.utils import load_img

dog_png = load_img('images/dog.png', target_size=(299,299))
incep_dog_png = inception_v3.preprocess_input( np.array( dog_png ) )

In [12]:
inception = keras.applications.InceptionV3()
predictions = inception.predict( incep_dog_png[ np.newaxis , : ] )
inception_v3.decode_predictions( predictions )

1/1 [==============================] - 2s 2s/step


[[('n02104029', 'kuvasz', 0.13920642),
  ('n02099712', 'Labrador_retriever', 0.07765269),
  ('n02106166', 'Border_collie', 0.07182899),
  ('n02111500', 'Great_Pyrenees', 0.066354565),
  ('n02099601', 'golden_retriever', 0.028421246)]]

In [13]:
cat_png = load_img('images/cat.png', target_size=(299,299))
incep_prep_cat = inception_v3.preprocess_input(np.array(cat_png))
predictions = inception.predict(incep_prep_cat[np.newaxis, : ])
inception_v3.decode_predictions( predictions )

1/1 [==============================] - 0s 17ms/step


[[('n02124075', 'Egyptian_cat', 0.6864776),
  ('n02123159', 'tiger_cat', 0.13308471),
  ('n02123045', 'tabby', 0.042191382),
  ('n04040759', 'radiator', 0.0016091414),
  ('n02971356', 'carton', 0.0011282415)]]

In [16]:
from keras.applications import inception_resnet_v2 as incep_res_v2

incep_res_pre_dog = incep_res_v2.preprocess_input( np.array(dog_png))
incep_resnet = keras.applications.InceptionResNetV2()
predictions = incep_resnet.predict(incep_res_pre_dog[np.newaxis, :])
incep_res_v2.decode_predictions( predictions )

1/1 [==============================] - 1s 1s/step


[[('n02099712', 'Labrador_retriever', 0.6557689),
  ('n02104029', 'kuvasz', 0.13987808),
  ('n02099601', 'golden_retriever', 0.055982478),
  ('n02111500', 'Great_Pyrenees', 0.049098987),
  ('n02100735', 'English_setter', 0.002121342)]]

In [17]:
incep_res_pre_cat = incep_res_v2.preprocess_input( np.array(cat_png))
incep_resnet = keras.applications.InceptionResNetV2()
predictions = incep_resnet.predict(incep_res_pre_cat[np.newaxis, :])
incep_res_v2.decode_predictions( predictions )

1/1 [==============================] - 1s 1s/step


[[('n02123045', 'tabby', 0.42519933),
  ('n02124075', 'Egyptian_cat', 0.25807074),
  ('n02123159', 'tiger_cat', 0.12800503),
  ('n02127052', 'lynx', 0.003449632),
  ('n04525038', 'velvet', 0.0024462815)]]